# Lesson 06 Tasks - Part 1: Normalization Practice (SOLUTIONS)

## Database Applications Development

**INSTRUCTOR VERSION - Complete Solutions**

## Setup: Import Libraries and Connect

In [ ]:
import sqlite3
import pandas as pd

# Connect to NBA database
conn = sqlite3.connect('nba_5seasons.db')
print("✅ Connected to NBA database!")

---
## Task 1: Understanding Primary Keys (10 points)

### Task 1a: View teams table structure

In [ ]:
# SOLUTION: View table structure using PRAGMA
query = """
PRAGMA table_info(teams);
"""

result = pd.read_sql(query, conn)
print(result)

### Task 1b: Identify the primary key

**Question:** What column is the primary key in the teams table?

**SOLUTION:** The primary key is `team_id`. You can tell because it has pk=1 in the PRAGMA output.

### Task 1c: Verify uniqueness

In [ ]:
# SOLUTION: Count total rows and unique values
query = """
SELECT 
    COUNT(*) as total_rows,
    COUNT(DISTINCT team_id) as unique_team_ids
FROM teams;
"""

result = pd.read_sql(query, conn)
print(result)
print("\nIf total_rows = unique_team_ids, then team_id is unique!")

### Task 1d: Explain why IDs are better than names

**SOLUTION:**
1. **Names can change** - Teams can rebrand or move cities (e.g., Charlotte Hornets → New Orleans Pelicans)
2. **IDs are guaranteed unique** - Two teams might have had similar names historically
3. **IDs are more efficient** - Numeric IDs are smaller and faster to search than long text strings
4. **BONUS:** IDs are unchanging - provide stable reference even if everything else changes

---
## Task 2: Understanding Composite Primary Keys (10 points)

### Task 2a: Find a duplicate game_id

In [ ]:
# SOLUTION: Show one game_id appearing twice (home and away teams)
query = """
SELECT season, game_id, team_id, matchup, pts
FROM team_game_stats
WHERE game_id = '0022100001'
ORDER BY team_id;
"""

result = pd.read_sql(query, conn)
print(result)
print("\nNotice: Same game_id appears twice (one for each team)")

### Task 2b: Explain composite key

**SOLUTION:** The table needs a composite key because:
- `game_id` alone is NOT unique - each game has 2 teams (home and away)
- `team_id` alone is NOT unique - each team plays 82 games
- `season` alone is NOT unique - multiple teams and games per season
- **The combination of all three IS unique** - identifies exactly one team's performance in one specific game in one specific season

### Task 2c: Verify composite key uniqueness

In [ ]:
# SOLUTION: Verify composite key uniqueness
query = """
SELECT 
    COUNT(*) as total_rows,
    COUNT(DISTINCT season || game_id || team_id) as unique_combinations
FROM team_game_stats;
"""

result = pd.read_sql(query, conn)
print(result)
print("\nIf total_rows = unique_combinations, the composite key is valid!")

---
## Task 3: Understanding Foreign Keys (10 points)

### Task 3a: Count games per team

In [ ]:
# SOLUTION: Join and count games
query = """
SELECT 
    t.team_id,
    t.full_name,
    COUNT(tgs.game_id) as games_played
FROM teams t
LEFT JOIN team_game_stats tgs ON t.team_id = tgs.team_id
WHERE tgs.season = '2021-22' OR tgs.season IS NULL
GROUP BY t.team_id, t.full_name
ORDER BY games_played DESC
LIMIT 10;
"""

result = pd.read_sql(query, conn)
print(result)

### Task 3b: Check for orphan records

In [ ]:
# SOLUTION: Check for orphan records
query = """
SELECT COUNT(*) as orphan_count
FROM team_game_stats tgs
LEFT JOIN teams t ON tgs.team_id = t.team_id
WHERE t.team_id IS NULL;
"""

result = pd.read_sql(query, conn)
print(result)
print("\nIf orphan_count = 0, referential integrity is maintained!")

### Task 3c: Explain foreign key benefit

**SOLUTION:** If we tried to insert a game stat for team_id = 999999:
- The database would **reject the insert** with a foreign key constraint error
- This **prevents orphan records** - game stats for teams that don't exist
- **Maintains data integrity** - every game must have a valid team
- Without foreign key constraints, we could insert invalid data that would break JOINs and queries

---
## Task 4: First Normal Form (1NF) (10 points)

### Task 4a: Check current tables for 1NF

In [ ]:
# SOLUTION: Show sample teams data
query = """
SELECT team_id, full_name, city, state
FROM teams
LIMIT 5;
"""

result = pd.read_sql(query, conn)
print(result)

### Task 4b: Identify 1NF satisfaction

**SOLUTION:** Yes, the teams table satisfies 1NF because:
- **All columns contain atomic values** - each cell has a single value (not a list)
- **No repeating groups** - no columns like player1, player2, player3
- Each column has one piece of information per row

### Task 4c: 1NF violation example

**SOLUTION - Example of 1NF Violation:**

```
team_id | team_name | players
--------|-----------|--------------------------------
1610612739 | Cavaliers | Garland, Mitchell, Mobley, Allen
```

**Why this violates 1NF:**
- The `players` column contains MULTIPLE values in one cell (a list)
- Not atomic - can't easily query "find teams with player X"
- Fixed limit - what if team has more players?
- Can't update individual players without parsing the whole string

**Correct 1NF design:**
```
team_id    | player_id
-----------|----------
1610612739 | 203507     <- One player per row
1610612739 | 203506
1610612739 | 203999
```

---
## Task 5: Understanding Data Redundancy (10 points)

### Task 5a: Calculate team name repetition

In [ ]:
# SOLUTION: Count Cavaliers games
query = """
SELECT COUNT(*) as cavaliers_games
FROM team_game_stats
WHERE team_id = 1610612739
  AND season = '2021-22';
"""

result = pd.read_sql(query, conn)
print(result)
print(f"\nIn denormalized design, 'Cleveland Cavaliers' would be stored {result['cavaliers_games'][0]} times!")

### Task 5b: Calculate storage savings

In [ ]:
# SOLUTION: Compare normalized vs denormalized storage

# Normalized: one team name per team
query_normalized = """
SELECT COUNT(*) as count FROM teams;
"""

# Denormalized: one team name per game
query_denormalized = """
SELECT COUNT(*) as count FROM team_game_stats WHERE season = '2021-22';
"""

normalized_count = pd.read_sql(query_normalized, conn)['count'][0]
denormalized_count = pd.read_sql(query_denormalized, conn)['count'][0]

print(f"Normalized: {normalized_count} team names stored")
print(f"Denormalized: {denormalized_count} team names stored")
print(f"Savings: {denormalized_count - normalized_count} duplicates eliminated!")

### Task 5c: Update anomaly example

**SOLUTION:**
- **Normalized:** 1 row (just update the teams table)
- **Denormalized:** ~2,460 rows (one for each game in the season)
- **Conclusion:** Normalized design is MUCH better because:
  - 2,460x faster to update
  - No risk of missing some rows
  - No risk of partial update if operation fails
  - Single source of truth for team name

---
## Task 6: Many-to-Many Relationships (Bonus +5 points)

### Task 6a: Find traded players

In [ ]:
# SOLUTION: Find players who played for multiple teams
query = """
SELECT 
    p.full_name,
    ps.season,
    COUNT(DISTINCT ps.team_id) as teams_count,
    GROUP_CONCAT(DISTINCT t.abbreviation) as teams
FROM player_season_stats ps
JOIN players p ON ps.player_id = p.player_id
JOIN teams t ON ps.team_id = t.team_id
WHERE ps.season = '2021-22'
GROUP BY p.player_id, p.full_name, ps.season
HAVING COUNT(DISTINCT ps.team_id) > 1
ORDER BY teams_count DESC
LIMIT 10;
"""

result = pd.read_sql(query, conn)
print(result)
print("\nThese players were traded during the season!")

### Task 6b: Explain junction table

**SOLUTION:** `player_season_stats` is a junction table because:
- It connects the **players** table to the **teams** table
- Enables a **many-to-many relationship**:
  - One player can play for multiple teams (over career or in one season if traded)
  - One team has multiple players
- Without a junction table, we couldn't represent this relationship properly
- Junction tables typically have foreign keys to both tables they connect (player_id → players, team_id → teams)

---
## Summary

**Key Concepts Covered:**
1. ✅ Primary keys (single and composite)
2. ✅ Foreign keys and referential integrity
3. ✅ First Normal Form (1NF)
4. ✅ Data redundancy problems
5. ✅ Update anomalies
6. ✅ Junction tables for M:N relationships

**Great work completing Part 1!**

In [ ]:
# Close connection
conn.close()
print("✅ Connection closed. Solutions complete!")